In [1]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
import numpy as np
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

In [2]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [5]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [6]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [7]:
def tokenize_lematize(tekst):
    tokens = word_tokenize(tekst.lower())  # vse besede pišemo z malo začetnico
    tagged = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
        if word.isalpha()    # znebimo se ločil, st,...
    ]

In [8]:
import random
from sklearn.feature_extraction import text

In [16]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()

custom_words = {
    'book', 'novel', 'story', 'reader',
    'edition', 'classic', 'introduction',
    'publish', 'note', 'cover', 'series',
    'time', 'year', 'new', 'make', 'tell',
    'begin', 'just', 'work', 'face',
    'place', 'mean', 'text', 'author', 'original', 'u', 'seller',
    'masterpiece', 'literature', 'best', 'read'
}

all_stopwords = list(base_stopwords.union(custom_words))

In [17]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)

#random.seed(42)
# vzamemo 150/200 knjig, ostale bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\leto_2003\knjige_03_opisi\*.txt')[:150]
#filepaths= random.sample(filepaths, 150)
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
# custom_stopwords = list(text.ENGLISH_STOP_WORDS.union({'book', 'novel', 'story', 'author', 'character',
#                                                   'edition', 'classic', 'bestseller', 'review',
#                                                   'read', 'reader', 'york', 'times', 'new'}))
vectorizer= CountVectorizer(stop_words= all_stopwords, #custom_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=1, 
                            max_df=0.9)

In [18]:
X = vectorizer.fit_transform(filepaths) 


c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['far'] not in stop_words.
  warnings.warn(


In [19]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 9697 stored elements and shape (150, 4363)>
  Coords	Values
  (0, 2694)	2
  (0, 1341)	1
  (0, 2315)	2
  (0, 4201)	4
  (0, 3244)	2
  (0, 1973)	1
  (0, 3752)	1
  (0, 4314)	1
  (0, 4328)	1
  (0, 100)	1
  (0, 943)	2
  (0, 1187)	2
  (0, 1157)	1
  (0, 3233)	1
  (0, 2712)	1
  (0, 2845)	1
  (0, 3704)	1
  (0, 3271)	1
  (0, 2923)	1
  (0, 3147)	1
  (0, 48)	1
  (0, 1760)	5
  (0, 4014)	1
  (0, 2108)	1
  (0, 569)	1
  :	:
  (149, 2835)	1
  (149, 1228)	2
  (149, 4324)	1
  (149, 347)	1
  (149, 3188)	1
  (149, 1297)	1
  (149, 1192)	1
  (149, 3554)	1
  (149, 3002)	1
  (149, 526)	1
  (149, 879)	1
  (149, 968)	1
  (149, 1151)	1
  (149, 671)	1
  (149, 1068)	1
  (149, 2091)	1
  (149, 1440)	1
  (149, 2172)	1
  (149, 662)	1
  (149, 306)	1
  (149, 3132)	1
  (149, 1696)	1
  (149, 191)	1
  (149, 2481)	1
  (149, 243)	1
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]
   abbreviate  abe

In [20]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Stevilo komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break

    return W, H, errors

In [23]:
# test 

W, H, errors = nmf(X, 3)
print(errors)
#print(W)
#print(H)

[np.float64(124.8705532121681), np.float64(124.1412678207591), np.float64(123.4892431875738), np.float64(122.6270662217047), np.float64(121.60200163406118), np.float64(120.91654892751988), np.float64(120.61389319505642), np.float64(120.47309686487172), np.float64(120.3897287447635), np.float64(120.33493514977191), np.float64(120.29332405292291), np.float64(120.25927262229973), np.float64(120.23469299417187), np.float64(120.21496034552585), np.float64(120.19796238201954), np.float64(120.18414976508897), np.float64(120.17368534579445), np.float64(120.16604277964892), np.float64(120.16014712244355), np.float64(120.15502419334675), np.float64(120.14963856782242), np.float64(120.14521052731237), np.float64(120.14171258296633), np.float64(120.13876345317934), np.float64(120.13623192454358), np.float64(120.13392964991182), np.float64(120.13176125609712), np.float64(120.12973228403693), np.float64(120.12781602353438), np.float64(120.12604492959612), np.float64(120.12448728891249), np.float64(1

In [24]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[-15:]]
    print(f"Tema {topic_idx +1}: {' '.join(top_words)}")

Tema 1: harvard truth police member discover murder priory curator clue paris secret neveu vinci da langdon
Tema 2: dark way girl child want mother young life men family woman man know world love
Tema 3: krakauer create america miller jazz like great faith american christ passion franklin live god life
